### NYC Parking Tickets Project — Advanced Solution

This notebook provides a production-style solution for the NYC parking ticket extract project.

It focuses on:

- lazy file reading
- typed named tuple records
- robust parsing and validation
- reusable analytics helpers
- one-pass summary analytics
- optional exports
- lightweight self-tests

The sample file should be named:

```python
nyc_parking_tickets_extract.csv
```

### Project goals

1. Create a lazy iterator that returns a named tuple for each row.
2. Convert values to appropriate Python types.
3. Calculate the number of violations by car make.

The solution keeps lazy evaluation wherever possible. Aggregations such as frequency counts need an accumulator, but the CSV rows themselves are still consumed one at a time.

In [1]:
import csv
from collections import Counter, namedtuple
from datetime import datetime, date
from itertools import islice
from pathlib import Path
from typing import Callable, Iterable, Iterator, Mapping, NamedTuple, Optional, Sequence, Tuple

### Configuration

Update `CSV_FILE` only if your file is stored somewhere else.

In [2]:
CSV_FILE = Path("nyc_parking_tickets_extract.csv")

EXPECTED_COLUMNS = (
    "Summons Number",
    "Plate ID",
    "Registration State",
    "Plate Type",
    "Issue Date",
    "Violation Code",
    "Vehicle Body Type",
    "Vehicle Make",
    "Violation Description",
)

DATE_FORMATS = (
    "%m/%d/%Y",  # common format in the sample extract
    "%Y-%m-%d",  # useful fallback for cleaner CSV exports
)

### Typed named tuple model

`typing.NamedTuple` creates a real named tuple while also documenting the expected Python type of each field.

In [3]:
class ParkingTicket(NamedTuple):
    summons_number: int
    plate_id: Optional[str]
    registration_state: Optional[str]
    plate_type: Optional[str]
    issue_date: date
    violation_code: int
    vehicle_body_type: Optional[str]
    vehicle_make: str
    violation_description: Optional[str]

### Parsing helpers

The helpers below keep data conversion centralized and testable.

Design choices:

- Empty text values become `None`.
- Vehicle make is normalized to uppercase.
- Missing vehicle make becomes `"UNKNOWN"` so the counting step is stable.
- Required integer and date fields fail fast with clear error messages.

In [4]:
def clean_text(value):
    """Return stripped text, or None when the value is empty."""
    if value is None:
        return None

    value = value.strip()
    if value == "":
        return None

    return value


def clean_upper(value):
    """Return normalized uppercase text, or None when the value is empty."""
    value = clean_text(value)
    if value is None:
        return None
    return value.upper()


def parse_int(value, field_name):
    """Parse a required integer field."""
    value = clean_text(value)

    if value is None:
        raise ValueError("Missing required integer value for {!r}".format(field_name))

    try:
        return int(value)
    except ValueError:
        raise ValueError("Invalid integer for {!r}: {!r}".format(field_name, value))


def parse_date(value, field_name, date_formats=DATE_FORMATS):
    """Parse a required date field using one of the accepted date formats."""
    value = clean_text(value)

    if value is None:
        raise ValueError("Missing required date value for {!r}".format(field_name))

    for date_format in date_formats:
        try:
            return datetime.strptime(value, date_format).date()
        except ValueError:
            pass

    raise ValueError(
        "Invalid date for {!r}: {!r}. Accepted formats: {}".format(
            field_name,
            value,
            ", ".join(date_formats),
        )
    )


def normalize_make(value, unknown_label="UNKNOWN"):
    """Normalize vehicle make for stable grouping and counting."""
    value = clean_upper(value)
    if value is None:
        return unknown_label
    return value

### Header validation

This protects the rest of the notebook from silently producing incorrect results when the wrong CSV file is used.

In [5]:
def validate_columns(actual_columns, expected_columns=EXPECTED_COLUMNS):
    """Validate that all required columns exist in the CSV header."""
    if actual_columns is None:
        raise ValueError("CSV file has no header row.")

    actual = tuple(actual_columns)
    expected = tuple(expected_columns)

    missing = [column for column in expected if column not in actual]

    if missing:
        raise ValueError(
            "CSV file is missing expected columns: {}".format(
                ", ".join(repr(column) for column in missing)
            )
        )

    return True

### Row parser

This function converts one raw `csv.DictReader` row into one typed `ParkingTicket`.

In [6]:
def parse_ticket_row(row):
    """Convert one raw CSV row mapping into a typed ParkingTicket named tuple."""
    return ParkingTicket(
        summons_number=parse_int(row.get("Summons Number"), "Summons Number"),
        plate_id=clean_text(row.get("Plate ID")),
        registration_state=clean_upper(row.get("Registration State")),
        plate_type=clean_upper(row.get("Plate Type")),
        issue_date=parse_date(row.get("Issue Date"), "Issue Date"),
        violation_code=parse_int(row.get("Violation Code"), "Violation Code"),
        vehicle_body_type=clean_upper(row.get("Vehicle Body Type")),
        vehicle_make=normalize_make(row.get("Vehicle Make")),
        violation_description=clean_text(row.get("Violation Description")),
    )

### Goal 1 — lazy iterator

`iter_parking_tickets` is a generator. It opens the file, validates the header, and yields one parsed named tuple at a time.

It also supports optional bad-row collection through the `errors` list. In normal strict mode, the iterator fails fast when malformed data is found. In tolerant mode, malformed rows are skipped and recorded.

In [7]:
def iter_parking_tickets(
    file_path=CSV_FILE,
    encoding="utf-8",
    skip_bad_rows=False,
    errors=None,
):
    """
    Lazily yield typed ParkingTicket records from the NYC parking ticket CSV.

    Parameters
    ----------
    file_path:
        Path to the CSV file.

    encoding:
        Text encoding used by the file. Use "utf-8-sig" if the file has a BOM.

    skip_bad_rows:
        If False, raise an exception on malformed rows.
        If True, skip malformed rows and optionally collect errors.

    errors:
        Optional list that receives tuples of:
        (row_number, raw_row, exception)

    Yields
    ------
    ParkingTicket
        One typed named tuple per valid CSV row.
    """
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(
            "Could not find {!s}. Put the CSV next to this notebook or update CSV_FILE.".format(path)
        )

    with path.open(mode="r", newline="", encoding=encoding) as csv_file:
        reader = csv.DictReader(csv_file)
        validate_columns(reader.fieldnames)

        # CSV row 1 is the header, so data rows start at line 2.
        for row_number, row in enumerate(reader, start=2):
            try:
                yield parse_ticket_row(row)
            except Exception as exc:
                if errors is not None:
                    errors.append((row_number, row, exc))

                if skip_bad_rows:
                    continue

                raise ValueError("Failed to parse CSV row {}".format(row_number))

### Preview without loading the whole file

`islice` reads only the first few parsed rows.

In [8]:
preview = list(islice(iter_parking_tickets(), 5))
preview

[ParkingTicket(summons_number=4006478550, plate_id='VAD7274', registration_state='VA', plate_type='PAS', issue_date=datetime.date(2016, 10, 5), violation_code=5, vehicle_body_type='4D', vehicle_make='BMW', violation_description='BUS LANE VIOLATION'),
 ParkingTicket(summons_number=4006462396, plate_id='22834JK', registration_state='NY', plate_type='COM', issue_date=datetime.date(2016, 9, 30), violation_code=5, vehicle_body_type='VAN', vehicle_make='CHEVR', violation_description='BUS LANE VIOLATION'),
 ParkingTicket(summons_number=4007117810, plate_id='21791MG', registration_state='NY', plate_type='COM', issue_date=datetime.date(2017, 4, 10), violation_code=5, vehicle_body_type='VAN', vehicle_make='DODGE', violation_description='BUS LANE VIOLATION'),
 ParkingTicket(summons_number=4006265037, plate_id='FZX9232', registration_state='NY', plate_type='PAS', issue_date=datetime.date(2016, 8, 23), violation_code=5, vehicle_body_type='SUBN', vehicle_make='FORD', violation_description='BUS LANE 

### Verify parsed field types

This is a quick sanity check that dates and integers are being converted properly.

In [9]:
if preview:
    first_ticket = preview[0]
    parsed_type_report = {
        field_name: type(getattr(first_ticket, field_name)).__name__
        for field_name in first_ticket._fields
    }
    parsed_type_report

### Generic lazy-friendly analytics helpers

These functions accept any iterable of `ParkingTicket` objects. That means they work with the lazy CSV iterator, filtered iterators, or small test lists.

In [10]:
def count_by(records, field_name):
    """Count records by any ParkingTicket field."""
    if field_name not in ParkingTicket._fields:
        raise ValueError(
            "Unknown field {!r}. Valid fields are: {}".format(
                field_name,
                ", ".join(ParkingTicket._fields),
            )
        )

    return Counter(getattr(record, field_name) for record in records)


def top_n(counter, n=10):
    """Return the top n values from a Counter."""
    return counter.most_common(n)


def filter_by_state(records, state):
    """Lazily yield records for a specific registration state."""
    state = clean_upper(state)

    for record in records:
        if record.registration_state == state:
            yield record


def filter_by_make(records, make):
    """Lazily yield records for a specific vehicle make."""
    make = normalize_make(make)

    for record in records:
        if record.vehicle_make == make:
            yield record


def filter_by_date_range(records, start_date=None, end_date=None):
    """
    Lazily yield records whose issue_date falls inside an inclusive date range.

    start_date and end_date can be date objects or strings in DATE_FORMATS.
    """
    if isinstance(start_date, str):
        start_date = parse_date(start_date, "start_date")

    if isinstance(end_date, str):
        end_date = parse_date(end_date, "end_date")

    for record in records:
        if start_date is not None and record.issue_date < start_date:
            continue

        if end_date is not None and record.issue_date > end_date:
            continue

        yield record

### Goal 2 — number of violations by car make

The file is still consumed lazily. The only object retained in memory is the `Counter` of makes.

In [11]:
violations_by_make = count_by(iter_parking_tickets(), "vehicle_make")
violations_by_make

Counter({'TOYOT': 112,
         'HONDA': 106,
         'FORD': 104,
         'CHEVR': 76,
         'NISSA': 70,
         'DODGE': 45,
         'FRUEH': 44,
         'ME/BE': 38,
         'GMC': 35,
         'HYUND': 35,
         'BMW': 34,
         'LEXUS': 26,
         'INTER': 25,
         'JEEP': 22,
         'NS/OT': 18,
         'SUBAR': 18,
         'INFIN': 13,
         'LINCO': 12,
         'CHRYS': 12,
         'ACURA': 12,
         'AUDI': 12,
         'VOLVO': 12,
         'MITSU': 11,
         'ISUZU': 10,
         'CADIL': 9,
         'KIA': 8,
         'VOLKS': 8,
         'HIN': 6,
         'KENWO': 5,
         'UNKNOWN': 5,
         'ROVER': 5,
         'BUICK': 5,
         'MAZDA': 5,
         'MERCU': 4,
         'JAGUA': 3,
         'SMART': 3,
         'PORSC': 3,
         'WORKH': 2,
         'SATUR': 2,
         'SCION': 2,
         'SAAB': 2,
         'HINO': 2,
         'FIR': 1,
         'OLDSM': 1,
         'PETER': 1,
         'CITRO': 1,
         'GEO': 1,
 

### Results sorted from most common to least common

In [12]:
violations_by_make.most_common()

[('TOYOT', 112),
 ('HONDA', 106),
 ('FORD', 104),
 ('CHEVR', 76),
 ('NISSA', 70),
 ('DODGE', 45),
 ('FRUEH', 44),
 ('ME/BE', 38),
 ('GMC', 35),
 ('HYUND', 35),
 ('BMW', 34),
 ('LEXUS', 26),
 ('INTER', 25),
 ('JEEP', 22),
 ('NS/OT', 18),
 ('SUBAR', 18),
 ('INFIN', 13),
 ('LINCO', 12),
 ('CHRYS', 12),
 ('ACURA', 12),
 ('AUDI', 12),
 ('VOLVO', 12),
 ('MITSU', 11),
 ('ISUZU', 10),
 ('CADIL', 9),
 ('KIA', 8),
 ('VOLKS', 8),
 ('HIN', 6),
 ('KENWO', 5),
 ('UNKNOWN', 5),
 ('ROVER', 5),
 ('BUICK', 5),
 ('MAZDA', 5),
 ('MERCU', 4),
 ('JAGUA', 3),
 ('SMART', 3),
 ('PORSC', 3),
 ('WORKH', 2),
 ('SATUR', 2),
 ('SCION', 2),
 ('SAAB', 2),
 ('HINO', 2),
 ('FIR', 1),
 ('OLDSM', 1),
 ('PETER', 1),
 ('CITRO', 1),
 ('GEO', 1),
 ('YAMAH', 1),
 ('BSA', 1),
 ('MINI', 1),
 ('PONTI', 1),
 ('SPRI', 1),
 ('PLYMO', 1),
 ('UPS', 1),
 ('FIAT', 1),
 ('UD', 1),
 ('UTILI', 1),
 ('GMCQ', 1),
 ('STAR', 1),
 ('AM/T', 1),
 ('MI/F', 1)]

### Additional value: top-N report

A small reporting helper makes the output easier to reuse.

In [13]:
def print_counter_report(counter, title, n=10):
    """Pretty-print the top n values from a Counter."""
    print(title)
    print("=" * len(title))

    for rank, (key, count) in enumerate(counter.most_common(n), start=1):
        print("{:>2}. {:<20} {:>6}".format(rank, str(key), count))


print_counter_report(violations_by_make, "Top vehicle makes by violations", n=10)

Top vehicle makes by violations
 1. TOYOT                   112
 2. HONDA                   106
 3. FORD                    104
 4. CHEVR                    76
 5. NISSA                    70
 6. DODGE                    45
 7. FRUEH                    44
 8. ME/BE                    38
 9. GMC                      35
10. HYUND                    35


### Additional value: one-pass analytics

The next function calculates several useful metrics in one pass over the file:

- total ticket count
- violations by vehicle make
- violations by registration state
- violations by violation code
- violations by issue month
- earliest and latest issue dates

This is efficient because the CSV is read once.

In [14]:
class ParkingTicketAnalytics(NamedTuple):
    total_records: int
    by_make: Counter
    by_state: Counter
    by_violation_code: Counter
    by_issue_month: Counter
    earliest_issue_date: Optional[date]
    latest_issue_date: Optional[date]


def issue_month_key(issue_date):
    """Return YYYY-MM string for grouping dates by month."""
    return "{:04d}-{:02d}".format(issue_date.year, issue_date.month)


def analyze_tickets(records):
    """Analyze parking ticket records in one pass."""
    total_records = 0
    by_make = Counter()
    by_state = Counter()
    by_violation_code = Counter()
    by_issue_month = Counter()
    earliest_issue_date = None
    latest_issue_date = None

    for record in records:
        total_records += 1

        by_make[record.vehicle_make] += 1
        by_state[record.registration_state] += 1
        by_violation_code[record.violation_code] += 1
        by_issue_month[issue_month_key(record.issue_date)] += 1

        if earliest_issue_date is None or record.issue_date < earliest_issue_date:
            earliest_issue_date = record.issue_date

        if latest_issue_date is None or record.issue_date > latest_issue_date:
            latest_issue_date = record.issue_date

    return ParkingTicketAnalytics(
        total_records=total_records,
        by_make=by_make,
        by_state=by_state,
        by_violation_code=by_violation_code,
        by_issue_month=by_issue_month,
        earliest_issue_date=earliest_issue_date,
        latest_issue_date=latest_issue_date,
    )


analytics = analyze_tickets(iter_parking_tickets())
analytics

ParkingTicketAnalytics(total_records=1000, by_make=Counter({'TOYOT': 112, 'HONDA': 106, 'FORD': 104, 'CHEVR': 76, 'NISSA': 70, 'DODGE': 45, 'FRUEH': 44, 'ME/BE': 38, 'GMC': 35, 'HYUND': 35, 'BMW': 34, 'LEXUS': 26, 'INTER': 25, 'JEEP': 22, 'NS/OT': 18, 'SUBAR': 18, 'INFIN': 13, 'LINCO': 12, 'CHRYS': 12, 'ACURA': 12, 'AUDI': 12, 'VOLVO': 12, 'MITSU': 11, 'ISUZU': 10, 'CADIL': 9, 'KIA': 8, 'VOLKS': 8, 'HIN': 6, 'KENWO': 5, 'UNKNOWN': 5, 'ROVER': 5, 'BUICK': 5, 'MAZDA': 5, 'MERCU': 4, 'JAGUA': 3, 'SMART': 3, 'PORSC': 3, 'WORKH': 2, 'SATUR': 2, 'SCION': 2, 'SAAB': 2, 'HINO': 2, 'FIR': 1, 'OLDSM': 1, 'PETER': 1, 'CITRO': 1, 'GEO': 1, 'YAMAH': 1, 'BSA': 1, 'MINI': 1, 'PONTI': 1, 'SPRI': 1, 'PLYMO': 1, 'UPS': 1, 'FIAT': 1, 'UD': 1, 'UTILI': 1, 'GMCQ': 1, 'STAR': 1, 'AM/T': 1, 'MI/F': 1}), by_state=Counter({'NY': 779, 'NJ': 94, 'PA': 30, 'FL': 14, 'VA': 10, 'CT': 10, 'IN': 7, 'MA': 6, 'TX': 5, 'TN': 4, '99': 4, 'SC': 3, 'MD': 3, 'AZ': 3, 'CA': 3, 'RI': 3, 'AL': 3, 'GA': 3, 'NC': 2, 'IA': 2, 'ME

### Additional value: compact analytics summary

In [15]:
print("Total records:", analytics.total_records)
print("Earliest issue date:", analytics.earliest_issue_date)
print("Latest issue date:", analytics.latest_issue_date)
print()

print_counter_report(analytics.by_make, "Top makes", n=10)
print()
print_counter_report(analytics.by_state, "Top registration states", n=10)
print()
print_counter_report(analytics.by_violation_code, "Top violation codes", n=10)

Total records: 1000
Earliest issue date: 2016-07-01
Latest issue date: 2017-06-27

Top makes
 1. TOYOT                   112
 2. HONDA                   106
 3. FORD                    104
 4. CHEVR                    76
 5. NISSA                    70
 6. DODGE                    45
 7. FRUEH                    44
 8. ME/BE                    38
 9. GMC                      35
10. HYUND                    35

Top registration states
 1. NY                      779
 2. NJ                       94
 3. PA                       30
 4. FL                       14
 5. VA                       10
 6. CT                       10
 7. IN                        7
 8. MA                        6
 9. TX                        5
10. TN                        4

Top violation codes
 1. 21                      162
 2. 36                      140
 3. 38                      121
 4. 14                       86
 5. 46                       54
 6. 37                       49
 7. 40                       

### Additional value: filtered analytics examples

These examples reuse the same lazy pipeline.

They are safe for large files because the filters are generators.

In [16]:
# Example 1: count violations by make for vehicles registered in NY only.
ny_make_counts = count_by(
    filter_by_state(iter_parking_tickets(), "NY"),
    "vehicle_make",
)

ny_make_counts.most_common(10)

[('TOYOT', 99),
 ('FORD', 83),
 ('HONDA', 81),
 ('CHEVR', 63),
 ('NISSA', 60),
 ('DODGE', 35),
 ('ME/BE', 32),
 ('GMC', 30),
 ('FRUEH', 27),
 ('HYUND', 27)]

In [17]:
# Example 2: count violations by state for TOYOTA vehicles only.
toyota_state_counts = count_by(
    filter_by_make(iter_parking_tickets(), "TOYOTA"),
    "registration_state",
)

toyota_state_counts.most_common(10)

[]

In [18]:
# Example 3: count vehicle makes within a date range.
# Update the dates to match values that exist in your extract.
date_range_make_counts = count_by(
    filter_by_date_range(
        iter_parking_tickets(),
        start_date=None,
        end_date=None,
    ),
    "vehicle_make",
)

date_range_make_counts.most_common(10)

[('TOYOT', 112),
 ('HONDA', 106),
 ('FORD', 104),
 ('CHEVR', 76),
 ('NISSA', 70),
 ('DODGE', 45),
 ('FRUEH', 44),
 ('ME/BE', 38),
 ('GMC', 35),
 ('HYUND', 35)]

### Additional value: export sorted results

This helper writes a `Counter` to a CSV file. It is useful when submitting or sharing the result.

In [19]:
def export_counter(counter, output_file, key_name="key", count_name="count"):
    """Export a Counter to CSV, sorted by count descending."""
    output_path = Path(output_file)

    with output_path.open(mode="w", newline="", encoding="utf-8") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow([key_name, count_name])

        for key, count in counter.most_common():
            writer.writerow([key, count])

    return output_path


exported_file = export_counter(
    violations_by_make,
    "violations_by_vehicle_make.csv",
    key_name="vehicle_make",
    count_name="violation_count",
)

exported_file

WindowsPath('violations_by_vehicle_make.csv')

### Optional pandas display

The core solution does not depend on pandas. This cell is only for a nicer notebook display.

In [20]:
try:
    import pandas as pd

    make_counts_df = pd.DataFrame(
        violations_by_make.most_common(),
        columns=["vehicle_make", "violation_count"],
    )

    display(make_counts_df.head(20))
except ImportError:
    print("pandas is not installed. Here are the top 20 results instead:")
    print(violations_by_make.most_common(20))

pandas is not installed. Here are the top 20 results instead:
[('TOYOT', 112), ('HONDA', 106), ('FORD', 104), ('CHEVR', 76), ('NISSA', 70), ('DODGE', 45), ('FRUEH', 44), ('ME/BE', 38), ('GMC', 35), ('HYUND', 35), ('BMW', 34), ('LEXUS', 26), ('INTER', 25), ('JEEP', 22), ('NS/OT', 18), ('SUBAR', 18), ('INFIN', 13), ('LINCO', 12), ('CHRYS', 12), ('ACURA', 12)]


### Lightweight self-tests

These tests validate the parser and helpers without needing the real CSV file.

In [21]:
def run_self_tests():
    sample_row = {
        "Summons Number": "1363745270",
        "Plate ID": "GZH7067",
        "Registration State": "NY",
        "Plate Type": "PAS",
        "Issue Date": "07/09/2015",
        "Violation Code": "20",
        "Vehicle Body Type": "SUBN",
        "Vehicle Make": "toyot",
        "Violation Description": "NO PARKING-DAY/TIME LIMITS",
    }

    ticket = parse_ticket_row(sample_row)

    assert isinstance(ticket, ParkingTicket)
    assert ticket.summons_number == 1363745270
    assert ticket.issue_date == date(2015, 7, 9)
    assert ticket.violation_code == 20
    assert ticket.vehicle_make == "TOYOT"

    records = [
        ticket,
        ticket._replace(vehicle_make="HONDA"),
        ticket._replace(vehicle_make="HONDA"),
    ]

    counts = count_by(records, "vehicle_make")
    assert counts["HONDA"] == 2
    assert counts["TOYOT"] == 1

    filtered = list(filter_by_make(records, "honda"))
    assert len(filtered) == 2

    print("All self-tests passed.")


run_self_tests()

All self-tests passed.


### Final reusable solution

For most use cases, call `solve()` and use the returned `Counter`.

In [22]:
def solve(file_path=CSV_FILE):
    """Return violations counted by vehicle make for the given CSV file."""
    return count_by(iter_parking_tickets(file_path), "vehicle_make")


solution = solve()
solution.most_common(10)

[('TOYOT', 112),
 ('HONDA', 106),
 ('FORD', 104),
 ('CHEVR', 76),
 ('NISSA', 70),
 ('DODGE', 45),
 ('FRUEH', 44),
 ('ME/BE', 38),
 ('GMC', 35),
 ('HYUND', 35)]

### Summary

This notebook satisfies the project requirements and adds reusable functionality:

- lazy row iteration
- typed named tuple records
- explicit data parsing
- schema validation
- flexible counters
- lazy filters
- one-pass analytics
- CSV export
- parser self-tests